# 03 — Feature engineering du contenu des films

**Objectif** : transformer ce que MovieLens dit d'un film — titre, année, genres, tags libres —
en représentations exploitables par un modèle.

MovieLens ne contient **aucune description de film**. On n'en invente pas : tout ce qui suit
repose sur les colonnes réellement présentes.

In [1]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from recsys.analysis import model_report, plots
from recsys.config import load_config

%matplotlib inline
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_colwidth", 90)

config = load_config()
processed = config.paths.processed_dir
movies = pd.read_parquet(processed / "movies.parquet")
tags = pd.read_parquet(processed / "tags.parquet")


def show(fig):
    display(fig)
    plt.close(fig)

from recsys.features.item_text import aggregate_tags, build_item_documents, content_coverage
from recsys.features.tfidf import build_tfidf_features

## 1. Agréger les tags

Un tag est un texte libre posé par un utilisateur sur un film. On compte, pour chaque couple
(film, tag), le nombre d'**utilisateurs distincts** : un tag posé par 300 personnes décrit mieux
un film qu'un tag posé une fois.

In [2]:
tag_counts = aggregate_tags(tags)
print(f"{len(tags):,} applications de tags -> {len(tag_counts):,} couples (film, tag)")
matrix_id = movies.loc[movies["title"] == "Matrix, The (1999)", "movieId"].iloc[0]
tag_counts[tag_counts["movieId"] == matrix_id].head(12).reset_index(drop=True)

1,093,333 applications de tags -> 463,458 couples (film, tag)


,movieId,tag,n_users
0,2571,sci-fi,244
1,2571,virtual reality,203
2,2571,dystopia,177
3,2571,philosophy,152
4,2571,cyberpunk,148
5,2571,artificial intelligence,144
6,2571,thought-provoking,143
7,2571,alternate reality,131
8,2571,martial arts,124
9,2571,action,120


In [3]:
per_tag = tag_counts.groupby("tag")["movieId"].nunique()
pd.DataFrame({
    "seuil (films décrits)": [1, 2, 3, 5, 10, 50],
    "tags conservés": [int((per_tag >= t).sum()) for t in [1, 2, 3, 5, 10, 50]],
    "part des couples (film, tag) conservés": [
        tag_counts["tag"].map(per_tag).ge(t).mean() for t in [1, 2, 3, 5, 10, 50]
    ],
})

,seuil (films décrits),tags conservés,"part des couples (film, tag) conservés"
0,1,65380,1.0000
1,2,27650,0.9186
2,3,19278,0.8825
3,5,12512,0.8330
4,10,6821,0.7533
5,50,1475,0.5204


La majorité des tags ne décrivent qu'un seul film : ils ne peuvent rapprocher aucune paire de
films. Le vocabulaire TF-IDF ne garde que les tags décrivant au moins
`content.tag_min_df` films.

## 2. Prévenir la fuite temporelle

Les tags sont horodatés. Pour évaluer un modèle à une date de coupure, on ne garde que les tags
posés **avant** cette date.

In [4]:
results = model_report.load_results(config.paths.reports_dir, "content_results.json")
rows = []
for name, fold in results["folds"].items():
    cutoff = pd.Timestamp(fold["train_end"])
    kept = aggregate_tags(tags, until=cutoff)
    rows.append({"jeu": name, "coupure": cutoff.date(),
                 "couples (film, tag) utilisables": len(kept),
                 "part du total": len(kept) / len(tag_counts),
                 "films avec tags": kept["movieId"].nunique()})
pd.DataFrame(rows)

,jeu,coupure,"couples (film, tag) utilisables",part du total,films avec tags
0,validation,2016-06-25,209695,0.4525,21030
1,test,2018-01-04,240254,0.5184,23966


## 3. Un document par film

In [5]:
documents = build_item_documents(movies, tag_counts, config.content.max_tags)
sample = documents[documents["title"].isin(
    ["Matrix, The (1999)", "Toy Story (1995)", "Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)"]
)]
for text in sample["text"]:
    print(text, end="\n\n")

Toy Story (1995). Genres: Adventure, Animation, Children, Comedy, Fantasy. Tags: pixar, animation, disney, tom hanks, computer animation, funny, children, witty, friendship, toys, comedy, family, animated, adventure, humorous.

Matrix, The (1999). Genres: Action, Sci-Fi, Thriller. Tags: sci-fi, virtual reality, dystopia, philosophy, cyberpunk, artificial intelligence, thought-provoking, alternate reality, martial arts, action, philosophical, post-apocalyptic, keanu reeves, surreal, post apocalyptic.

Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001). Genres: Comedy, Romance. Tags: beautifully filmed, quirky, feel-good, surreal, whimsical, great soundtrack, romance, stylized, paris, atmospheric, visually appealing, french, comedy, fairy tale, notable soundtrack.



In [6]:
pd.Series(content_coverage(documents)).to_frame("catalogue complet (62 423 films)")

,catalogue complet (62 423 films)
n_movies,"62,423.0000"
share_with_genres,0.9189
share_with_tags,0.7249
share_with_year,0.9934
share_title_only,0.0466
median_tags_when_tagged,5.0000


In [7]:
pd.DataFrame(results["content_coverage"]).rename(
    columns={"validation": "films évaluables (validation)", "test": "films évaluables (test)"}
)

,films évaluables (validation),films évaluables (test)
n_movies,"13,980.0000","16,007.0000"
share_with_genres,0.9993,0.9972
share_with_tags,0.9196,0.9068
share_with_year,1.0000,0.9992
share_title_only,0.0001,0.0007
median_tags_when_tagged,8.0000,8.0000


Le catalogue complet est mal couvert par les tags, mais les films qui ont assez de notes pour
entrer dans l'évaluation le sont presque tous : les films sans contenu sont surtout des films
que personne ne note.

## 4. Représentation TF-IDF par champ

Trois blocs (genres, tags, titre), chacun avec son IDF, pondérés puis concaténés.

In [8]:
features = build_tfidf_features(
    documents, tag_counts, genre_weight=0.5, tag_weight=1.0, title_weight=0.3,
    tag_min_df=config.content.tag_min_df,
)
kinds = pd.Series(features.feature_names).str.split(":").str[0].value_counts()
print(f"Matrice : {features.matrix.shape[0]:,} films x {features.matrix.shape[1]:,} termes, "
      f"{features.matrix.nnz / features.matrix.shape[0]:.1f} termes non nuls par film en moyenne")
kinds.rename("termes").to_frame().T

Matrice : 62,423 films x 31,462 termes, 10.1 termes non nuls par film en moyenne


,tag,title,genre
termes,19278,12165,19


In [9]:
def top_terms(title, n=10):
    row = features.matrix[documents.index[documents["title"] == title][0]].tocoo()
    order = np.argsort(-row.data)[:n]
    return pd.DataFrame({"terme": features.feature_names[row.col[order]], "poids": row.data[order]})


display(top_terms("Matrix, The (1999)"))
display(top_terms("Toy Story (1995)"))

,terme,poids
0,genre:Sci-Fi,0.2873
1,title:matrix,0.2592
2,genre:Action,0.2341
3,genre:Thriller,0.2219
4,tag:virtual reality,0.1412
5,tag:keanu reeves,0.1305
6,tag:cyberpunk,0.1260
7,tag:artificial intelligence,0.1246
8,tag:philosophy,0.1239
9,tag:hackers,0.1237


,terme,poids
0,tag:pixar,0.2547
1,title:toy,0.2179
2,genre:Fantasy,0.2146
3,genre:Animation,0.2109
4,genre:Children,0.2108
5,tag:tom hanks,0.2053
6,tag:toys,0.1976
7,genre:Adventure,0.1929
8,tag:computer animation,0.1816
9,tag:disney,0.1738


L'IDF fait son travail : un tag rare et consensuel (`cyberpunk`, `pixar`) pèse plus qu'un genre
partagé par des milliers de films.

## 5. Ce que ces features ne contiennent pas

- pas de synopsis, de casting ni de réalisateur (absents de MovieLens) ;
- pas de variables numériques par film (popularité, note moyenne, ancienneté) : elles serviront
  au modèle de ranking du Sprint 5, où elles seront calculées sur la seule période
  d'entraînement.